In [ ]:
import pandas as pd
from sklearn.datasets import make_classification

N_SAMPLES = 1_000_000
N_FEATURES = 100
N_INFORMATIVE = 26
N_CLASSES = 2
RANDOM_STATE = 42

# Job parameters are passed in through notebook task base_parameters.
dbutils.widgets.text("catalog", "flip_flopper")
dbutils.widgets.text("schema", "default")
dbutils.widgets.text("table_name", "generated_data")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
table_name = dbutils.widgets.get("table_name").strip()

if not catalog or not schema or not table_name:
    raise ValueError("catalog, schema, and table_name must all be provided")


def build_pandas_frame() -> pd.DataFrame:
    x, y = make_classification(
        n_samples=N_SAMPLES,
        n_features=N_FEATURES,
        n_informative=N_INFORMATIVE,
        n_redundant=0,
        n_repeated=0,
        n_classes=N_CLASSES,
        random_state=RANDOM_STATE,
    )
    feature_cols = [f"feature_{i:03d}" for i in range(N_FEATURES)]
    frame = pd.DataFrame(x, columns=feature_cols)
    frame["label"] = y.astype("int64")
    return frame


spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
full_table = f"{catalog}.{schema}.{table_name}"
pandas_df = build_pandas_frame()
spark_df = spark.createDataFrame(pandas_df)
spark_df.write.mode("overwrite").saveAsTable(full_table)

print(f"Wrote {spark_df.count()} rows to {full_table}")
